# 05 — Follower Count Regression

Self-contained notebook: no `%run` dependency on project scripts.

In [ ]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import yaml

# In Colab, change only this path after mounting Google Drive.
PROJECT_ROOT = Path("..").resolve() if Path.cwd().name == "src" else Path(".").resolve()
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
with CONFIG_PATH.open(encoding="utf-8") as handle:
    CONFIG = yaml.safe_load(handle)

def project_path(key):
    return PROJECT_ROOT / CONFIG["paths"][key]

OUTPUT = project_path("output")
PROCESSED = project_path("processed")
for folder in [OUTPUT / "reports", OUTPUT / "figures", OUTPUT / "models", OUTPUT / "labeling", OUTPUT / "logs", PROCESSED]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)

This cross-sectional model describes associations with current Follower_Count_k. It is not a causal model or a future-growth forecast.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

shops = pd.read_csv(PROCESSED / "shops_clean.csv")
features = ["Years_Active", "Total_Products", "Rating_Average", "Chat_Response_Rate_pct"]
target = "Follower_Count_k"
data = shops.dropna(subset=[target, *features])
if len(data) < 5:
    raise ValueError(f"Need at least 5 complete shops; found {len(data)}")
x_train, x_test, y_train, y_test = train_test_split(data[features], data[target], test_size=0.2, random_state=CONFIG["project"]["random_state"])
model = LinearRegression().fit(x_train, y_train)
prediction = model.predict(x_test)

In [ ]:
x_scaler, y_scaler = StandardScaler(), StandardScaler()
standardized_model = LinearRegression().fit(x_scaler.fit_transform(data[features]), y_scaler.fit_transform(data[[target]]).ravel())
coefficients = pd.DataFrame({
    "Feature": features,
    "Coefficient": model.coef_,
    "Direction": np.where(model.coef_ >= 0, "positive", "negative"),
    "Absolute_Coefficient": np.abs(model.coef_),
    "Standardized_Coefficient": standardized_model.coef_,
})
metrics = {"MAE": mean_absolute_error(y_test, prediction), "RMSE": mean_squared_error(y_test, prediction) ** 0.5, "R2": r2_score(y_test, prediction) if len(y_test) >= 2 else None}
coefficients.to_csv(OUTPUT / "reports" / "Regression_Result.csv", index=False)
with (OUTPUT / "reports" / "regression_metrics.json").open("w", encoding="utf-8") as handle:
    json.dump(metrics, handle, indent=2)
display(coefficients)
display(pd.DataFrame([metrics]))